In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
# import torch

# base_model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
# adapter_path = "grpo_output_final"

# tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto"
# )

# model = PeftModel.from_pretrained(base_model, adapter_path)

# prompt = "Explain AI like I am 5 years old."
# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# with torch.no_grad():
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=100,
#         do_sample=True,
#         temperature=0.7,
#         top_p=0.9
#     )

# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## GRPO Final

In [19]:
# !pip install -q datasets transformers trl peft accelerate bitsandbytes

import re
import torch
from difflib import SequenceMatcher
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer

In [20]:
data = [
    {
        "prompt": "Explain reinforcement learning in simple terms.",
        "reference_answer": (
            "Reinforcement learning is a type of machine learning where an agent learns "
            "by taking actions, receiving rewards or penalties, and improving over time."
        ),
    },
    {
        "prompt": "Write a polite email for delayed delivery.",
        "reference_answer": (
            "We apologize for the delay in your delivery. Your order is on the way, "
            "and we appreciate your patience and understanding."
        ),
    },
    {
        "prompt": "What is a neural network?",
        "reference_answer": (
            "A neural network is a machine learning model inspired by the human brain. "
            "It learns patterns from data using layers of connected nodes."
        ),
    },
    {
        "prompt": "Explain overfitting in ML.",
        "reference_answer": (
            "Overfitting happens when a model learns the training data too closely, "
            "including noise, and performs poorly on new unseen data."
        ),
    },
]

dataset = Dataset.from_list(data)

In [21]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [22]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)


In [23]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def similarity_score(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

def correctness_reward(prompts, completions, reference_answer, **kwargs):
    rewards = []
    for completion, ref in zip(completions, reference_answer):
        text = completion if isinstance(completion, str) else str(completion)
        sim = similarity_score(text, ref)
        rewards.append(float(sim * 5.0))   # scale to 0–5
    return rewards

def helpfulness_reward(prompts, completions, **kwargs):
    rewards = []
    bad_phrases = [
        "i don't know",
        "no idea",
        "maybe",
        "not sure",
        "random magic",
    ]
    for completion in completions:
        text = completion if isinstance(completion, str) else str(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Prefer complete and reasonably detailed answers
        if 40 <= len(text) <= 220:
            score += 1.0

        # Prefer sentence-like responses
        if "." in text or "," in text:
            score += 0.5

        # Penalize obviously unhelpful phrases
        if any(bp in text_norm for bp in bad_phrases):
            score -= 1.5

        rewards.append(float(score))
    return rewards

def clarity_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else str(completion)
        score = 0.0

        # Penalize overly short or overly long outputs
        if len(text) < 20:
            score -= 1.0
        elif len(text) > 300:
            score -= 0.5
        else:
            score += 0.5

        # Reward readable structure a little
        if re.search(r"[A-Za-z]", text):
            score += 0.5

        rewards.append(float(score))
    return rewards

In [24]:
training_args = GRPOConfig(
    output_dir="grpo_output",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=2,
    num_train_epochs=1,
    logging_steps=1,
    report_to=[],
    remove_unused_columns=False,
)

In [25]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, helpfulness_reward, clarity_reward],
    args=training_args,
    train_dataset=dataset,
)

In [26]:
trainer.train()

Step,Training Loss
1,-0.701511
2,-0.701557
3,0.000004
4,0.056630


TrainOutput(global_step=4, training_loss=-0.3366083651781082, metrics={'train_runtime': 66.4785, 'train_samples_per_second': 0.06, 'train_steps_per_second': 0.06, 'total_flos': 0.0, 'train_loss': -0.3366083651781082})

In [27]:
trainer.save_model("grpo_output_final")
tokenizer.save_pretrained("grpo_output_final")

('grpo_output_final/tokenizer_config.json',
 'grpo_output_final/chat_template.jinja',
 'grpo_output_final/tokenizer.json')

In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
# import torch

# base_model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
# adapter_path = "grpo_output_final"

# tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# base_model = AutoModelForCausalLM.from_pretrained(
#     base_model_id,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto"
# )

# model = PeftModel.from_pretrained(base_model, adapter_path)

# prompt = "Explain AI in simple terms."

# inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# with torch.no_grad():
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=100,
#         do_sample=True,
#         temperature=0.7
#     )

# print(tokenizer.decode(outputs[0], skip_special_tokens=True))